prepatring for model

In [67]:
import pandas as pd
real_data = pd.read_csv('../datasets/pvAmpSeq_simulation_ready.csv')
real_data

,patient_id,sample_id,episode,marker,allele,frequency,country,MOIs
0,Sero-040,Sero-040-R01,1.0,Chr01,Chr01-1,0.531524,ethiopia,6
1,Sero-182,Sero-182-R12,3.0,Chr11,Chr11-2,0.149320,ethiopia,3
2,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-1,0.729045,ethiopia,3
3,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-2,0.129575,ethiopia,3
4,Sero-182,Sero-182-R12,3.0,Chr14,Chr14-1,0.931269,ethiopia,3
...,...,...,...,...,...,...,...,...
6734,AR-322,AR-322-140,2.0,Chr03,Chr03-1,0.526098,solomon_island,2
6735,AR-322,AR-322-140,2.0,Chr02,Chr02-2,0.148837,solomon_island,2
6736,AR-322,AR-322-140,2.0,Chr02,Chr02-6,0.042156,solomon_island,2
6737,AR-302,AR-302-0,2.0,Chr11,Chr11-2,0.232755,solomon_island,2


In [62]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
from DeepSets import DataPreprocessor
preprocessed_data, x = (real_data 
                     .pipe(DataPreprocessor.removing_single_episode_patients)
                     .pipe(DataPreprocessor.calculate_population_Allele_frequencies, population='country')
                     .pipe(DataPreprocessor.calculate_MOIs)
                     .pipe(DataPreprocessor.paired_episode_assigner)
                    #  .pipe(DataPreprocessor.build_deepsets_tensors, mode='prediction')

)


In [61]:
result['results_table'].columns

Index(['pair_id', 'prob_C', 'prob_L', 'prob_I', 'predicted_class',
       'sample_id_paired', 'patient_id', 'true_episode'],
      dtype='object')

In [63]:
from DeepSets import run_prediction

result = run_prediction(real_data, mode='prediction')
result['results_table']

143


D:\Educational\PvRecNet\backend\PvDeepSetNet\src\DeepSets\Model_result.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../src/DeepSets

,pair_id,patient_id,true_episode,prob_C,prob_L,prob_I,predicted_class
0,AR-024_P1,AR-024,1.0,3.471411e-07,0.304471,0.695529,I
1,AR-024_P1,AR-024,2.0,3.471411e-07,0.304471,0.695529,I
2,AR-024_P10,AR-024,5.0,3.380386e-09,0.173901,0.826099,I
3,AR-024_P10,AR-024,4.0,3.380386e-09,0.173901,0.826099,I
4,AR-024_P2,AR-024,1.0,9.909202e-04,0.999007,0.000002,L
...,...,...,...,...,...,...,...
783,Sero-277_P1,Sero-277,1.0,1.636915e-06,0.992661,0.007338,L
784,Sero-281_P1,Sero-281,1.0,2.164313e-08,0.304484,0.695516,I
785,Sero-281_P1,Sero-281,2.0,2.164313e-08,0.304484,0.695516,I
786,Sero-288_P1,Sero-288,1.0,3.749500e-08,0.155791,0.844209,I


In [45]:
preprocessed_data[['sample_id_paired', 'patient_id', 'true_episode']].drop_duplicates().reset_index(drop=True)

,sample_id_paired,patient_id,true_episode
0,AR-024_P1,AR-024,1.0
1,AR-024_P1,AR-024,2.0
2,AR-024_P2,AR-024,1.0
3,AR-024_P2,AR-024,3.0
4,AR-024_P3,AR-024,4.0
...,...,...,...
783,Sero-277_P1,Sero-277,1.0
784,Sero-281_P1,Sero-281,1.0
785,Sero-281_P1,Sero-281,2.0
786,Sero-288_P1,Sero-288,1.0


In [15]:
from DeepSets import run_prediction

result = run_prediction(real_data, mode='prediction')

11
394
5


In [7]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from DeepSets import DataPreprocessor, build_deepsets_tensors


def run_Prediction(data, mode='prediction'):

    # perform basic preprocess
    preprocessed_tensor = (
        data
        .pipe(DataPreprocessor.removing_single_episode_patients)
        .pipe(DataPreprocessor.calculate_population_Allele_frequencies, population='country')
        .pipe(DataPreprocessor.calculate_MOIs)
        .pipe(DataPreprocessor.paired_episode_assigner)
        .pipe(DataPreprocessor.build_deepsets_tensors, mode=mode)
    )
    loaded_data = DataPreprocessor.load_tensor(preprocessed_tensor)
    return loaded_data

result = run_Prediction(real_data)

11
394
5


In [3]:
import json
# reading allele dictionary
with open('allele_dict.json', 'r') as file:
    allele_to_id = json.load(file)

In [4]:
import torch
from DeepSets import PvDeepSets
# 1. Define the device
device = torch.device("cpu")

# 2. Initialize the model with the same parameters used during training
# Make sure len(allele_to_id) matches the training setup!
n_alleles = len(allele_to_id) 
model = PvDeepSets(n_alleles=n_alleles)
print(n_alleles)
# 3. Load the weights (state_dict)
# Replace "pv_deepsets_weights.pth" with your actual filename
model.load_state_dict(torch.load("../src/DeepSets/pv_deepsets_weights20260402.pth", map_location=device))



143


C:\Users\sinel\AppData\Local\Temp\ipykernel_24916\2347922982.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../src/DeepSets/pv_deepse

<All keys matched successfully>

In [5]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

def get_model_results(model, dataloader, device, threshold=0.5):
    model.eval()
    all_predicted = []
    y_pred_filtered = []
    undetermined_count = 0

    with torch.no_grad():
        for X_alleles, allele_mask, marker_mask, priors, MOI in dataloader:
            # ... (your existing GPU transfer code) ...
            y_pred = model(X_alleles, allele_mask, marker_mask, priors, MOI)
            
            # Apply Softmax if your model doesn't already output probabilities
            # probs = torch.softmax(y_pred, dim=1) 
            probs = y_pred 

            # Get max probabilities and their indices
            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(max_probs)):
                if max_probs[i] >= threshold:
                    y_pred_filtered.append(preds[i].item())
                else:
                    undetermined_count += 1

            all_predicted.append(y_pred.cpu().numpy())

    print(f"Total undetermined samples removed: {undetermined_count}")
    return np.vstack(all_predicted), y_pred_filtered

predicted_probs, y_pred_hard = get_model_results(model, result, device)




Total undetermined samples removed: 1


In [40]:
y_pred_hard

[2,
 2,
 1,
 0,
 2,
 2,
 2,
 2,
 0,
 2,
 1,
 2,
 2,
 1,
 1,
 0,
 0,
 0,
 0,
 2,
 1,
 2,
 1,
 0,
 2,
 2,
 2,
 1,
 0,
 1,
 2,
 2,
 1,
 2,
 2,
 2,
 0,
 0,
 0,
 1,
 2,
 0,
 1,
 1,
 2,
 1,
 0,
 1,
 2,
 2,
 2,
 1,
 2,
 2,
 2,
 0,
 2,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 0,
 0,
 2,
 1,
 0,
 0,
 2,
 0,
 2,
 2,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 2,
 2,
 0,
 2,
 2,
 0,
 2,
 1,
 1,
 2,
 2,
 2,
 2,
 0,
 2,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 0,
 2,
 2,
 2,
 0,
 2,
 2,
 2,
 2,
 2,
 0,
 1,
 0,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 2,
 2,
 1,
 2,
 2,
 1,
 1,
 2,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 1,
 2,
 1,
 2,
 2,
 2,
 2,
 1,
 2,
 1,
 1,
 2,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 1,
 2,
 2,
 2,
 1,
 2,
 1,
 1,
 2,
 2,
 1,
 2,
 1,
 1,
 2,
 2,
 2,
 2,
 1,
 2,
 2,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 2,
 2,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,


In [ ]:
# Step 1: removing patients with only one episode
# Step 2: make all possible pair (including making episode 1&2)
# step 3: Calculate MOIs
# step 4: Calculate Afreq